# 01c. Thresholding

## Preliminaries

A fundamental analysis in neural engineering is to determine the threshold stimulus required to elicit activation in a neuron. Differences in threshold between neurons can provide insights into their functional properties, and can often help explain outcomes in neuromodulation related to efficacy and side-effect. This notebook demonstrates how to perform thresholding analyses using AxonML. 

Thresholding requires 5 things:

1. A neuron model that can spike.
2. A field that encodes the coupling strength between the neuron and the extracellular stimulus.
3. A 'unit' stimulus that represents the extracellular stimulus at a unit amplitude.
4. A criterion to determine activation.
5. A tolerance for the threshold search.

Let $F(I, w, f) \in {0, 1}$ indicate activation given stimulus amplitude $I$, unit waveform $w$ and field $f$. 'Threshold' then defines the value $I_{\theta}$ such that $F(I_{\theta}, w, f) = 1$ and $F(I, w, f) = 0$ for all $I < I_{\theta}$ (within tolerance). AxonML implements a bisection search algorithm to efficiently find $I_{\theta}$ for batches of fibers, fields, and stimuli.

We've seen previously how to create spiking neuron models, fields, and stimuli (see [here](01a_extracellular_stim_and_rec.ipynb)). We will use the same principles here.

In [ ]:
# To start, we will use a simple myelinated axon with Hodgkin-Huxley dynamics.
import axonml as ax
from axonml.models.mod import hh
from axonml.units import mm, um

# define the model
model = ax.Myelinated(
    diameters=[6.0], n_node=201, node_length=1.0, celsius=6.3, v_init=-65.0
)
model.insert(hh)

# define the field
field = ax.isotropic_point(z=200.0)(model)

# define the waveform
waveform = ax.mono_rect(amp=-1.0, pw=0.1, delay=0.5)

# define the criterion
criterion = ax.callbacks.Active(threshold=0.0, node_check=[5, -5])

This will check nodes 5 and -5 for activation, defined as at least one node crossing 0 mV custom criteria can also be defined by subclassing ax.callbacks.ThresholdCallback for example, ax.callbacks.ActiveAL is another built-in criterion that determines activation based on at least some specified number of nodes crossing threshold.

We then use the {py:class}`~axonml.models.instruments.thresholder.Thresholder` instrument to perform the threshold search. This instrument takes as input the neuron model, field, stimulus, activation criterion, and tolerance, and returns the threshold values.

In [ ]:
from axonml.models.instruments import Thresholder
thresholder = Thresholder(
    model, criterion, field, waveform, ub=0.1, rtol=0.01
) # find threshold with 1% relative tolerance, initial upper bound 0.1 mA

ub, lb = thresholder.calculate_thresholds(tstop=5.0, dt=0.005)
print(f"Threshold found between {lb.item():.4f} mA and {ub.item():.4f} mA")

The industry standard for mammalian myelinated axons is the McIntyre-Richardson-Grill (MRG) model [1]. This and many other neuron models are available in the `axonml-models` package. All models expose the same core interfaces, so switching between models is straightforward.

In [ ]:
from axonml_models.models import bigMRG

model = bigMRG(diameters=[6.0], n_node=201)
field = ax.isotropic_point(z=200.0)(model)
waveform = ax.mono_rect(amp=-1.0, pw=0.1, delay=0.5)
node_indices = model.find('node', as_list=True)
criterion = ax.callbacks.Active(
    threshold=0.0, node_check=[node_indices[5], node_indices[-5]]
) # check nodes 5 and -5
thresholder = Thresholder(
    model, criterion, field, waveform, ub=0.1, rtol=0.01
)
ub, lb = thresholder.calculate_thresholds(tstop=5.0, dt=0.005)
print(f"Threshold found between {lb.item():.4f} mA and {ub.item():.4f} mA")


Activation threshold and fiber diameter are closely related; larger diameter fibers have lower thresholds. Here we will create a batch of MRG fibers with diameters ranging from 5.7 (the smallest diameter in the original MRG paper) to 20 microns, and compute their activation thresholds to a simple monophasic cathodic pulse.

In [ ]:
from axonml_models.models import bigMRG
import torch
import matplotlib.pyplot as plt

diameters = torch.linspace(5.7, 20.0, steps=10) * um
model = bigMRG(diameters=diameters, n_node=201).double()
field = ax.isotropic_point(z=200.0)(model)
waveform = ax.mono_rect(amp=-1.0, pw=0.02, delay=0.5) # shorter pulse width of 0.02 ms
node_indices = model.find('node', as_list=True)
criterion = ax.callbacks.Active(
    threshold=0.0, node_check=[node_indices[5], node_indices[-5]]
) # check nodes 5 and -5
thresholder = Thresholder(
    model, criterion, field, waveform, ub=0.1, rtol=0.01
)
ub, _ = thresholder.calculate_thresholds(tstop=5.0, dt=0.005)

plt.figure(figsize=(5,5), dpi=300)
plt.plot(diameters, ub.squeeze(), marker='o', linewidth=2.0)
plt.xlabel("Fiber Diameter (um)")
plt.ylabel("Activation Threshold (mA)")
plt.title("Activation Threshold vs Fiber Diameter for MRG Fibers")
plt.show()


[1] McIntyre, C. C., Richardson, A. G., & Grill, W. M. (2002). Modeling the excitability of mammalian nerve fibers: influence of afterpotentials on the recovery cycle. Journal of neurophysiology, 87(2), 995-1006.